In [1]:
import gc
import json
import time
import threading
import random

import rclpy
from rclpy.node import Node
from rclpy.action import ActionClient
from rclpy.executors import MultiThreadedExecutor
from std_msgs.msg import String
from geometry_msgs.msg import Pose
from geographic_msgs.msg import GeoPoint
from lotusim_msgs.msg import VesselCmd, VesselCmdArray, VesselPositionArray
from lotusim_msgs.msg import MASCmd as MASCmdMsg, VesselPositionArray
from lotusim_msgs.action import MASCmd, MASCmdArray

The messages (VesselCmd, VesselCmdArray) are in the interface folder. Please open it. 

In VesselCmd msg, the cmd is in a json string format as LOTUSim aims to be modular and the actuator msg are not catered to any physics engine and meant to be generic

The physics interface developed will then read the `cmd_string` and parse it to fit the physics engine you choose
```
std_msgs/Header header

string vessel_name

uint16 entity

# cmd in json string format
string cmd_string
```

Each variable is treated like a class variable.

Below are example of publisher, subscriber using xdyn as a physics engine.

Xdyn consume propeller command using propeller_name(rpm), propeller_name(P/D), where P/D stands for pitch to diameter ratio. For more information regarding xdyn, read the xdyn developer doc  

In [2]:
SPAWN_LATITUDE = 1.2605794416293148
SPAWN_LONGITUDE = 103.7516212463379
SPAWN_ALTITUDE = 0.0
OFFSET = 0.0001

class ExampleNode(Node):
    def __init__(self):
        super().__init__('example_control_node')
        self.pose_subscription = self.create_subscription(
            VesselPositionArray,
            '/lotusim/poses',
            self.poses_callback,
            10)
        self.vessel_poses= {}
        self.cmd_publisher = self.create_publisher(
            VesselCmdArray,
            '/lotusim/vessel_cmd_array',
            10
        )
        self.mas_action_client = ActionClient(self, MASCmd, '/lotusim/mas_cmd')
        self.vessel_id = 0 # ID number for the number of model in the simulation. Used in randomizing location spawned

    def poses_callback(self, msg):
        for vessel_position in msg.vessels:
            lat = vessel_position.geo_point.latitude
            long = vessel_position.geo_point.longitude
            self.vessel_poses[vessel_position.vessel_name] = (lat, long)

    def spawn_ship_with_dynamics(self):
        msg = MASCmdMsg()
        msg.cmd_type =  MASCmdMsg.CREATE_CMD
        msg.model_name =  "lrauv"
        msg.vessel_name =  "lrauv_" + str(self.vessel_id)        
        latlong_msg = GeoPoint()
        latlong_msg.latitude  = SPAWN_LATITUDE  + OFFSET * self.vessel_id * random.choice([-1, 1])
        latlong_msg.longitude = SPAWN_LONGITUDE + OFFSET * self.vessel_id * random.choice([-1, 1])
        latlong_msg.altitude = -30.0
        msg.geo_point = latlong_msg
        
        msg.sdf_string = """
        <lotus_param>
            <render_interface>
                <publish_render>true</publish_render>
                <renderer_type_name>lrauv</renderer_type_name>
            </render_interface>
            <physics_engine_interface>
            <underwater>
                <connection_type>XDynWebSocket</connection_type>
                <uri>ws://127.0.0.1:12346</uri>
                <thrusters>
                    <thrusters1>propeller</thrusters1>
                </thrusters>
            </underwater>
            <surface>
                <connection_type>XDynWebSocket</connection_type>
                <uri>ws://127.0.0.1:12345</uri>
                <thrusters>
                    <thrusters1>propeller</thrusters1>
                </thrusters>
            </surface>
            <init_state>Underwater</init_state>
            </physics_engine_interface>
        </lotus_param>
        """
        self.vessel_id+=1
        goal_msg = MASCmd.Goal()
        goal_msg.cmd = msg
        self.mas_action_client.wait_for_server()
        return self.mas_action_client.send_goal_async(goal_msg)

    def send_propeller_command(self, vessel_name:str, propeller_name: str, rpm: float, pd: float):
        """
        Prepare and send a propeller command message.

        Args:
            rpm (float): Propeller rotation speed in RPM.
            pd (float): Propeller pitch-to-diameter ratio.
        """
        # Create a VesselCmdArray message
        cmd_array = VesselCmdArray()
        cmd = VesselCmd()
        cmd.vessel_name = vessel_name 
        cmd.cmd_string = json.dumps({"propeller_name(rpm)": rpm, "propeller_name(P/D)": pd})
        cmd_array.cmds.append(cmd)

        # Publish the message
        self.cmd_publisher.publish(cmd_array)

        # Also log with ROS logger
        print(f"{cmd.vessel_name} - propeller command published: rpm={rpm}, P/D={pd}")

In [3]:
stop_flag = False

if not rclpy.ok(): 
    rclpy.init(args=None)

try:
    stop_executor()  
except NameError:
    pass 
except Exception as e:
    print(f"Could not destroy resources: {e}")

node = ExampleNode()
executor = MultiThreadedExecutor()
executor.add_node(node)

def spin_executor():
    executor.spin()
def stop_executor():
    """Call this function to stop the spinning thread"""
    try:
        stop_flag = True
        executor.shutdown()
        spin_thread.join(timeout=2.0)
        node.destroy_node()
        time.sleep(2)
        print("Executor stopped successfully")
        
    except Exception as e:
        print(f"Error stopping executor: {e}")

spin_thread = threading.Thread(target=spin_executor, daemon=True)
spin_thread.start()

print("Node spinning in background thread")

In [4]:
node.spawn_ship_with_dynamics()

from IPython.display import clear_output
import datetime

while not stop_flag:
    clear_output(wait=True)
    print(f"[{datetime.datetime.now().strftime('%H:%M:%S')}]")
    for name, (lat, lon) in node.vessel_poses.items():
        print(f"  {name}: lat={lat:.6f}, lon={lon:.6f}")
    time.sleep(1)

In [ ]:
node.send_propeller_command("lrauv_1", "propeller", 200, 0.88)

In [5]:
stop_executor()